# Notebook 01 — Data Exploration

This notebook loads the Sign Language MNIST dataset, explores class distributions,
visualises sample images, and analyses the difficulty of the classification task.

**Prerequisites:** Run `python data/download_dataset.py` first.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import ASLDataLoader
from src.utils.config import load_config

sns.set_theme(style='whitegrid')
cfg = load_config('../config.yaml')
print('Config loaded:', cfg['project']['name'])

## 1. Load Dataset

In [ ]:
loader = ASLDataLoader(data_dir='../data/raw')
X_train, X_val, X_test, y_train, y_val, y_test = loader.load_and_split()

print(f'Training samples : {X_train.shape[0]}')
print(f'Validation samples: {X_val.shape[0]}')
print(f'Test samples      : {X_test.shape[0]}')
print(f'Image shape       : {X_train.shape[1:]}')
print(f'Number of classes : {len(loader.class_names)}')
print(f'Classes           : {loader.class_names}')

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, split, y, title in [
    (axes[0], 'train', y_train, 'Training Set'),
    (axes[1], 'test',  y_test,  'Test Set'),
]:
    unique, counts = np.unique(y, return_counts=True)
    ax.bar([loader.class_names[i] for i in unique], counts, color='steelblue')
    ax.set_title(f'Class Distribution — {title}')
    ax.set_xlabel('ASL Letter')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('../results/plots/class_distribution.png', dpi=150)
plt.show()
print('Saved class_distribution.png')

## 3. Sample Images per Class

In [ ]:
loader.visualize_samples(X_train, y_train, n_samples=3)
plt.savefig('../results/plots/sample_images.png', dpi=150)
plt.show()

## 4. Pixel Intensity Statistics

In [ ]:
print(f'Pixel range  : [{X_train.min():.4f}, {X_train.max():.4f}]')
print(f'Mean         : {X_train.mean():.4f}')
print(f'Std          : {X_train.std():.4f}')

plt.figure(figsize=(8, 4))
plt.hist(X_train.flatten(), bins=50, color='steelblue', edgecolor='white')
plt.title('Pixel Intensity Distribution (Training Set)')
plt.xlabel('Pixel Value')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('../results/plots/pixel_distribution.png', dpi=150)
plt.show()

## 5. Confusion-Prone Classes (Similarity Analysis)

In [ ]:
# Compute mean image per class and find most similar pairs via pixel correlation
num_classes = len(loader.class_names)
mean_imgs = []
for c in range(num_classes):
    idx = np.where(y_train == c)[0]
    mean_imgs.append(X_train[idx].mean(axis=0).flatten())

mean_imgs = np.array(mean_imgs)
corr_matrix = np.corrcoef(mean_imgs)

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix,
            xticklabels=loader.class_names,
            yticklabels=loader.class_names,
            cmap='RdYlGn', vmin=-1, vmax=1, annot=False)
plt.title('Mean-Image Correlation Between Classes')
plt.tight_layout()
plt.savefig('../results/plots/class_similarity.png', dpi=150)
plt.show()
print('High correlation = visually similar classes (potential confusion sources)')

## Summary

- Dataset loaded successfully with train/val/test splits.
- Classes are roughly balanced; minor imbalance will be handled via class weights.
- Some sign pairs (e.g. M/N, R/U/V) are visually similar — attention mechanisms may help distinguish them.
- Images are normalised to [0, 1] — no further preprocessing needed for CNN models.
- Transfer learning models will need up-scaling to at least 96×96 pixels.
